In [ ]:
"""Workbooks to analyze metadata."""
# pylint: disable=import-error, redefined-outer-name, unused-import

In [ ]:
%load_ext autoreload
%autoreload 2

### SETUP

In [ ]:
from __future__ import annotations

from collections import Counter, defaultdict
from pathlib import Path
from typing import DefaultDict, Dict, List

import pandas as pd
from IPython.display import display

from epiclass.core.metadata import Metadata, UUIDMetadata
from epiclass.utils.general_utility import (
    write_hdf5_paths_to_file,
    write_signal_ids_to_file,
)
from epiclass.utils.modify_metadata import filter_by_pairs
from epiclass.utils.notebooks.paper.paper_utilities import (
    ASSAY,
    ASSAY_ORDER,
    BIOMATERIAL_TYPE,
    CANCER,
    CELL_TYPE,
    DISEASE,
    EPIATLAS_16_CT,
    LIFE_STAGE,
    SEX,
    TRACK,
    MetadataHandler,
)

In [ ]:
CORE7_ASSAYS = ASSAY_ORDER[:7]

In [ ]:
ASSAY_MERGE_DICT: Dict[str, str] = {
    "rna_seq": "rna",
    "mrna_seq": "rna",
    "wgbs-pbat": "wgbs",
    "wgbs-standard": "wgbs",
}

In [ ]:
paper_dir = Path.home() / "Projects/epiclass/output/paper"
paper_meta_dir = paper_dir / "data" / "metadata"

In [ ]:
base = Path().home() / "Projects/epiclass/input/metadata"
# path = base / "dfreeze-v2" / "hg38_2023-epiatlas-dfreeze-pospurge-nodup_filterCtl.json"
path = base / "dfreeze-v1.0" / "hg38_2023-epiatlas_dfreeze_formatted_JR.json"
# path = base / "dfreeze-v1.0" / "hg38_2023-epiatlas_dfreeze_plus_encode_noncore_formatted_JR.json"
# path = base / "dfreeze-v2" / "hg38_2023-epiatlas-dfreeze_v2.1_w_encode_noncore_2.json"
my_metadata = Metadata(path)
meta_df = my_metadata.to_df()

In [ ]:
# force lowercase
for category in [ASSAY, BIOMATERIAL_TYPE, CELL_TYPE, DISEASE, LIFE_STAGE]:
    # print(meta_df[category].unique())
    # print({k: k.lower() for k in meta_df[category].unique() if isinstance(k, str)})
    my_metadata.convert_classes(
        category,
        converter={
            k: k.lower() for k in meta_df[category].unique() if isinstance(k, str)
        },
    )
    meta_df[category] = meta_df[category].str.lower()

In [ ]:
def display_gen_info(metadata: Metadata):
    """Display track type, assay and cell type class counts."""
    metadata.display_labels("track_type")
    metadata.display_labels(ASSAY)
    metadata.display_labels(CELL_TYPE)
    metadata.display_labels(SEX)
    # metadata.display_labels(CANCER)
    # metadata.display_labels(DISEASE)
    # metadata.display_labels(LIFE_STAGE)
    metadata.display_labels(TRACK)

In [ ]:
def count_trios(metadata: Metadata) -> Counter:
    """
    Count the occurrences of unique (track_type, assay, cell_type) trios in the metadata.

    Returns:
        Counter: A Counter object of the unique trios.
    """
    trios = Counter(
        [(dset["track_type"], dset[ASSAY], dset[CELL_TYPE]) for dset in metadata.datasets]
    )
    return trios

In [ ]:
def count_pairs_w_assay(metadata: Metadata, category: str) -> DefaultDict[str, Counter]:
    """
    Count the occurrences of each cell type for each assay in the dataset.

    Returns:
        defaultdict(Counter): A defaultdict of Counter objects with the count of cell types per assay.
    """
    pair_count = defaultdict(Counter)
    for dset in metadata.datasets:
        assay, other_label = dset[ASSAY], dset[category]
        pair_count[assay].update([other_label])
    return pair_count


def select_cell_types(metadata: Metadata, n=70) -> DefaultDict[str, List]:
    """
    Determines which cell types are needed to attain n datasets, for a given assay.
    Starts with T cell and then selects the most common cell types.

    Args:
        metadata (Metadata): A Metadata object containing dataset metadata.
        n (int, optional): Maximum number of cell types to select for each assay. Defaults to 70.

    Returns:
        defaultdict(list): A defaultdict with selected cell types for each assay.
    """
    cell_count = count_pairs_w_assay(metadata, CELL_TYPE)

    selected_ct = defaultdict(list)
    for assay, counter in cell_count.items():
        selected_ct[assay].append("T cell")
        i = min(counter["T cell"], n)
        del counter["T cell"]
        while i < n and counter:
            for cell_type, count in counter.most_common():
                i += min(count, n - i)
                selected_ct[assay].append(cell_type)
                del counter[cell_type]
                break
        if i < n:
            print(f"There is not at least {n} files for {assay}. Final number={i}")

    return selected_ct

In [ ]:
my_metadata.display_labels(BIOMATERIAL_TYPE)